In [0]:
from pyspark.sql import functions as F

clientes = spark.table("clientes_raw")

In [0]:
clientes = (
    clientes
    .withColumn(
        "faixa_idade",
        F.when(F.col("idade") < 30, "18_29")
        .when(F.col("idade") < 45, "30_44")
        .when(F.col("idade") < 60, "45_59")
        .otherwise("60_plus")
    )
    .withColumn(
        "faixa_renda",
        F.when(F.col("renda_mensal") < 3000, "baixa")
        .when(F.col("renda_mensal") < 8000, "media")
        .when(F.col("renda_mensal") < 20000, "alta")
        .otherwise("premium")
    )
    .withColumn(
        'log_vlr_fraude',
        F.log1p('valor_fraude')
    )
    .withColumn(
        'pix_por_renda',
        F.col("valor_total_pix_6m") / (F.col("renda_mensal") + 1)
    )
    .withColumn(
        'uso_digital_score',
        F.col("qtd_login_app_30d") + F.col("media_qtd_pix_6m")
    )
    .withColumn(
        "stress_cliente",
        F.col("qtd_protocolos_atendimento_90d") +
        (F.col("reclamacao_bacen") * 3) +
        (F.col("reclamacao_consumidor_gov") * 2)
    )
    .withColumn(
        "sem_estorno_flag",
        F.when(F.col("teve_estorno") == 0, 1).otherwise(0)
    )
    .withColumn(
        "valor_cliente_score",
        F.col("qtd_produtos") * 2 +
        (F.col("segmento").isin("Alta renda", "Private")).cast("int") * 5
    )
)

In [0]:
display(clientes.limit(10))

In [0]:
clientes.write.mode("overwrite").saveAsTable("clientes_features")